# Problema 7. Clasificación temporal y política de alertas

- Datos: `generar_vuelos(n=9600, seed=2026)`, sin modificar su mecanismo.
- Etiqueta: $Y = \mathbb 1\{\texttt{min\_retraso} \ge 15\}$. `min_retraso` no es predictor.
- `fecha_prediccion` solo se usa para ordenar y partir.
- No se usan las probabilidades internas del generador.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

SEED = 2026
GAP = 24          # una fila por hora: 24 filas = un día (latencia máxima de la etiqueta)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)

### Generador del taller (copiado sin cambios)

In [ ]:
def generar_vuelos(n=9600, seed=SEED):
    rng = np.random.default_rng(seed)
    fecha = pd.date_range('2024-01-01', periods=n, freq='h')
    hora = ((fecha + pd.Timedelta(hours=2)).hour).to_numpy()
    progreso = np.linspace(0, 1, n)
    aerolinea = rng.choice(['Aero_A', 'Aero_B', 'Aero_C'], n, p=[0.50, 0.30, 0.20])
    origen = rng.choice(['BOG', 'MDE', 'CLO', 'CTG'], n)
    distancia = rng.uniform(200, 1800, n)
    lluvia = rng.gamma(shape=1.2, scale=2.0, size=n)
    viento = np.clip(rng.normal(15, 6, n), 0, 45)
    ocupacion = np.clip(rng.normal(77, 12, n), 25, 100)
    trafico = np.clip(rng.normal(45 + 10 * progreso, 13, n), 5, 100)
    # Probabilidad del simulador: combina factores conocidos antes de la salida.
    logit = (
        -5.3 + 1.1 * (trafico - 45) / 13
        + 0.8 * (lluvia - 2.4) / 2.2 + 0.7 * (viento - 15) / 6
        + 0.35 * (ocupacion - 77) / 12 + 0.55 * (hora >= 17)
        + 0.40 * (origen == 'BOG') + 0.45 * (aerolinea == 'Aero_C')
    )
    probabilidad_generadora = 1 / (1 + np.exp(-logit))
    # La etiqueta se sortea: vuelos similares pueden tener resultados distintos.
    positivo = rng.binomial(1, probabilidad_generadora)
    retraso = np.where(positivo == 1, rng.integers(15, 181, n), rng.integers(-10, 15, n))
    datos = pd.DataFrame({
        'fecha_prediccion': fecha, 'hora_salida': hora,
        'distancia_km': distancia, 'lluvia_pronosticada_mm': lluvia,
        'viento_pronosticado_kmh': viento, 'ocupacion_programada_pct': ocupacion,
        'trafico_programado': trafico, 'aerolinea': aerolinea,
        'origen': origen, 'min_retraso': retraso,
    })
    # Los faltantes afectan a predictores, nunca a la etiqueta observada.
    for columna in ['lluvia_pronosticada_mm', 'viento_pronosticado_kmh',
                    'ocupacion_programada_pct', 'aerolinea']:
        datos.loc[rng.random(n) < 0.06, columna] = np.nan
    return datos

In [ ]:
df = generar_vuelos(n=9600, seed=SEED)
NUMERICAS = ["hora_salida", "distancia_km", "lluvia_pronosticada_mm",
             "viento_pronosticado_kmh", "ocupacion_programada_pct", "trafico_programado"]
CATEGORICAS = ["aerolinea", "origen"]
FEATURES = NUMERICAS + CATEGORICAS
df.shape

---
## (a) Diccionario, auditoría y partición temporal

### a.1 Diccionario: ¿qué se conoce dos horas antes de la salida?

La predicción se emite en `fecha_prediccion`, dos horas antes de la salida programada (`hora_salida` es la hora de `fecha_prediccion + 2 h`). Un predictor es válido solo si su valor existe en ese momento.

In [ ]:
diccionario = pd.DataFrame([
    ("fecha_prediccion", "datetime", "orden y partición", "Sí",
     "Instante en que se emite la predicción. No entra al modelo: solo ordena y parte."),
    ("hora_salida", "entero 0–23", "predictor numérico", "Sí",
     "Hora programada de salida (= hora de la predicción + 2 h); parte del itinerario."),
    ("distancia_km", "real", "predictor numérico", "Sí",
     "Distancia de la ruta; fija en el itinerario."),
    ("lluvia_pronosticada_mm", "real", "predictor numérico", "Sí",
     "Pronóstico, no lluvia observada: existe antes de la salida. Puede faltar."),
    ("viento_pronosticado_kmh", "real", "predictor numérico", "Sí",
     "Pronóstico de viento. Puede faltar."),
    ("ocupacion_programada_pct", "real", "predictor numérico", "Sí",
     "Ocupación según reservas a la fecha. Puede faltar."),
    ("trafico_programado", "real", "predictor numérico", "Sí",
     "Tráfico planificado en la franja; conocido por la programación."),
    ("aerolinea", "categórica", "predictor categórico", "Sí",
     "Operador del vuelo. Puede faltar; puede aparecer una categoría nueva."),
    ("origen", "categórica", "predictor categórico", "Sí",
     "Aeropuerto de origen. Puede aparecer una categoría nueva."),
    ("min_retraso", "entero", "solo construye la etiqueta", "No",
     "Se observa después de la salida: usarlo sería fuga de la respuesta."),
    ("retrasado", "binaria", "etiqueta Y", "No (llega en < 24 h)",
     "1{min_retraso ≥ 15}. Disponible menos de un día después de la salida."),
], columns=["variable", "tipo", "rol", "conocida 2 h antes", "justificación"])
assert set(FEATURES) == set(diccionario.loc[diccionario["rol"].str.startswith("predictor"), "variable"])
display(diccionario)

### a.2 Partición temporal

Se ordena por `fecha_prediccion`. Cortes en el 60 % y el 80 % de las filas; se excluyen las 24 filas inmediatamente anteriores a cada corte. Una fila es una hora y las etiquetas tardan menos de un día: sin el hueco, el final de un período tendría etiquetas que aún no se conocen al empezar el siguiente.

El `id_vuelo` es la posición tras ordenar (`V0000`–`V9599`).

In [ ]:
def separar_por_tiempo(datos, gap=GAP):
    ordenados = datos.sort_values("fecha_prediccion", kind="stable").reset_index(drop=True).copy()
    ordenados.insert(0, "id_vuelo", [f"V{i:04d}" for i in range(len(ordenados))])
    ordenados["retrasado"] = (ordenados["min_retraso"] >= 15).astype(int)
    c1, c2 = int(len(ordenados) * 0.60), int(len(ordenados) * 0.80)
    partes = {"entrenamiento": ordenados.iloc[:c1 - gap],
              "excluidas_1": ordenados.iloc[c1 - gap:c1],
              "validación": ordenados.iloc[c1:c2 - gap],
              "excluidas_2": ordenados.iloc[c2 - gap:c2],
              "prueba": ordenados.iloc[c2:]}
    return ordenados, {k: v.copy() for k, v in partes.items()}

ordenados, partes = separar_por_tiempo(df)
train, valid, test = partes["entrenamiento"], partes["validación"], partes["prueba"]

registro = pd.DataFrame([
    {"conjunto": k, "n": len(v),
     "filas": f"{v.index.min()}–{v.index.max()}",
     "id_inicio": v["id_vuelo"].iloc[0], "id_fin": v["id_vuelo"].iloc[-1],
     "fecha_inicio": v["fecha_prediccion"].min(), "fecha_fin": v["fecha_prediccion"].max()}
    for k, v in partes.items()])
display(registro)

Comprobación de la separación temporal (sin consultar etiquetas ni prevalencia de prueba).

In [ ]:
X_train, y_train = train[FEATURES], train["retrasado"]
X_valid, y_valid = valid[FEATURES], valid["retrasado"]
X_test, y_test = test[FEATURES], test["retrasado"]      # reservada hasta el inciso final

# 1. Orden estricto y hueco de al menos un día entre períodos
hueco_1 = valid["fecha_prediccion"].min() - train["fecha_prediccion"].max()
hueco_2 = test["fecha_prediccion"].min() - valid["fecha_prediccion"].max()
assert hueco_1 > pd.Timedelta(hours=GAP) and hueco_2 > pd.Timedelta(hours=GAP)

# 2. Conjuntos disjuntos y cobertura completa (con las filas excluidas)
ids = [set(v["id_vuelo"]) for v in partes.values()]
assert all(a.isdisjoint(b) for i, a in enumerate(ids) for b in ids[i + 1:])
assert sum(len(s) for s in ids) == len(df)

# 3. Ningún predictor es la respuesta ni el reloj
assert not {"min_retraso", "retrasado", "fecha_prediccion", "id_vuelo"} & set(FEATURES)

# 4. La fecha ya estaba ordenada y es regular (una fila por hora)
assert ordenados["fecha_prediccion"].is_monotonic_increasing
assert (ordenados["fecha_prediccion"].diff().dropna() == pd.Timedelta(hours=1)).all()

print(f"Hueco entrenamiento → validación: {hueco_1}")
print(f"Hueco validación → prueba:        {hueco_2}")

### a.3 Auditoría de entrenamiento

Solo con `train`. Validación y prueba no se describen aquí.

In [ ]:
auditoria = pd.DataFrame({
    "tipo": X_train.dtypes.astype(str),
    "faltantes": X_train.isna().sum(),
    "prop_faltante": X_train.isna().mean(),
    "n_únicos": X_train.nunique(),
})
display(auditoria)
print(f"Filas de entrenamiento con al menos un faltante: {X_train.isna().any(axis=1).mean():.2%}")
print(f"Prevalencia de retrasos en entrenamiento: {y_train.mean():.2%}  ({y_train.sum()} positivos de {len(y_train)})")

In [ ]:
display(X_train[NUMERICAS].describe(percentiles=[.01, .25, .5, .75, .99]).T)
for c in CATEGORICAS:
    display(X_train[c].value_counts(dropna=False, normalize=True).rename(f"proporción {c}").to_frame())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.flat, NUMERICAS):
    ax.hist(X_train[c].dropna(), bins=30, color="#2563eb", alpha=0.8)
    ax.set_title(c, fontsize=9)
for ax, c in zip(axes.flat[len(NUMERICAS):], CATEGORICAS):
    X_train[c].fillna("NaN").value_counts().sort_index().plot.bar(ax=ax, color="#2563eb")
    ax.set_title(c, fontsize=9)
fig.suptitle("Distribuciones en entrenamiento")
plt.tight_layout()
plt.show()

**Estabilidad dentro del entrenamiento.** Como los modelos se compararán con folds temporales, conviene ver si las variables se desplazan en el tiempo. Se compara la media de cada numérica entre el primer y el último cuarto del entrenamiento, en desviaciones estándar.

In [ ]:
q = len(X_train) // 4
inicio, final = X_train.iloc[:q][NUMERICAS], X_train.iloc[-q:][NUMERICAS]
deriva = ((final.mean() - inicio.mean()) / X_train[NUMERICAS].std()).rename("Δ media / DE")
display(deriva.to_frame())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(train["fecha_prediccion"], train["trafico_programado"].rolling(24 * 7).mean())
ax.set(title="trafico_programado en entrenamiento (media móvil de 7 días)", ylabel="tráfico")
plt.tight_layout()
plt.show()

**Hallazgos de la auditoría (solo entrenamiento, 5736 filas).**
- **Faltantes:** cerca del 6 % en `lluvia_pronosticada_mm`, `viento_pronosticado_kmh`, `ocupacion_programada_pct` y `aerolinea`; el 22.9 % de las filas tiene al menos uno. `hora_salida`, `distancia_km`, `trafico_programado` y `origen` están completas. La etiqueta no tiene faltantes.
- **Tipos:** seis numéricas (`hora_salida` es entera de 0 a 23 y cíclica, aunque aquí se trata como numérica) y dos categóricas (3 aerolíneas, 4 orígenes).
- **Distribuciones:** `lluvia` es asimétrica a la derecha (asimetría ≈ 1.8); `ocupacion` está truncada en 100 (2.5 % de las filas en el tope); `viento` tiene una pequeña masa en 0 (0.6 %). `distancia` y `hora_salida` son aproximadamente uniformes.
- **Desbalance:** 2.44 % de retrasos (140 positivos). Con tan pocos positivos, accuracy no sirve para comparar modelos.
- **Deriva:** `trafico_programado` sube ≈ 0.38 DE entre el primer y el último cuarto del entrenamiento; las demás numéricas son estables (< 0.1 DE). Es una razón más para comparar modelos con folds temporales y no aleatorios.

### a.4 Preparación ajustada dentro de cada entrenamiento

- Numéricas: imputar mediana → estandarizar.
- Categóricas: imputar moda → one-hot con `handle_unknown="ignore"` (una categoría nueva queda en ceros).
- Todo vive en un `Pipeline`: las medianas, modas, escalas y categorías se aprenden en `fit`, es decir, solo con las filas del entrenamiento que reciba (el conjunto completo o cada fold temporal).

In [ ]:
def construir_prep():
    numerico = Pipeline([("imputar", SimpleImputer(strategy="median")),
                         ("escalar", StandardScaler())])
    categorico = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                           ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", numerico, NUMERICAS),
                              ("cat", categorico, CATEGORICAS)], remainder="drop")

def construir_pipeline(estimador=None):
    estimador = LogisticRegression(max_iter=1000) if estimador is None else estimador
    return Pipeline([("prep", construir_prep()), ("modelo", estimador)])

# Prueba de robustez, solo con filas de entrenamiento
pipe = construir_pipeline().fit(X_train, y_train)
caso = X_train.iloc[[0, 1]].copy()
caso.loc[caso.index[0], ["lluvia_pronosticada_mm", "viento_pronosticado_kmh",
                         "ocupacion_programada_pct", "aerolinea"]] = np.nan   # todos los faltantes
caso.loc[caso.index[1], ["aerolinea", "origen"]] = ["Aero_NUEVA", "PEI"]       # categorías nuevas
p = pipe.predict_proba(caso)[:, 1]
assert np.isfinite(p).all()
print("Probabilidades (faltantes / categorías nuevas):", p.round(4))

# Lo aprendido proviene solo de entrenamiento
imp = pipe.named_steps["prep"].named_transformers_["num"].named_steps["imputar"]
assert np.allclose(imp.statistics_, X_train[NUMERICAS].median())
print("Categorías aprendidas:", pipe.named_steps["prep"].named_transformers_["cat"]
      .named_steps["codificar"].categories_)

---
## (b) Logística $L_2$ sin ponderar frente a logística $L_2$ con pesos de clase

**Objetivos** (el intercepto $b$ nunca se penaliza):

$$F_{\text{sin}}(w,b) = \frac{1}{N}\sum_n \ell_n + \frac{\lambda_2}{2}\|w\|_2^2, \qquad
F_{\text{pond}}(w,b) = \frac{1}{\sum_n a_{y_n}}\sum_n a_{y_n}\,\ell_n + \frac{\lambda_2}{2}\|w\|_2^2,$$

con $\ell_n = \log(1+e^{z_n}) - y_n z_n$, $z_n = x_n^\top w + b$ y $a_c = N/(2N_c)$ calculado en **cada** entrenamiento (cada pliegue y el entrenamiento completo).

**Normalización.** $\sum_n a_{y_n} = N_0\frac{N}{2N_0} + N_1\frac{N}{2N_1} = N$. Normalizar por $\sum_n a_{y_n}$ equivale a dividir por $N$, así que $\lambda_2$ tiene la misma escala en los dos modelos y la penalización es idéntica.

**Implementación.**
- Sin ponderar: `LogisticaGD` del Problema 4 (descenso por gradiente con paso $1/L$).
- Ponderado: se amplía esa clase. Pérdida y gradiente se multiplican por $a_{y_n}/\sum a$; la constante de Lipschitz pasa a $\tfrac14\|\operatorname{diag}(\sqrt{a})\tilde X\|_2^2/\sum a + \lambda_2$.

**Equivalencia con scikit-learn.** `LogisticRegression` minimiza $C\sum_n s_n\ell_n + \tfrac12\|w\|^2$ sin penalizar el intercepto. Multiplicando $F_{\text{pond}}$ por $1/\lambda_2$ se obtiene ese objetivo con $s_n = a_{y_n}$ (que es `class_weight="balanced"`) y $C = 1/(\lambda_2 N)$. Mismo minimizador, mismo tratamiento del intercepto; se verifica numéricamente abajo.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (average_precision_score, log_loss, brier_score_loss,
                             roc_curve, roc_auc_score, precision_recall_curve, auc,
                             precision_score, recall_score)
from src.modelos import LogisticaGD, log1pexp, _validar_Xy, _aumentar

LAMBDAS = (1e-3, 1e-2, 1e-1, 1.0, 10.0)
TOL_GD, MAX_ITER_GD = 1e-7, 300_000


class LogisticaPonderadaGD(LogisticaGD):
    # Logística L2 con pesos de clase a_c = N / (2 N_c), pérdida normalizada por sum(a).

    def fit(self, X, y):
        X, y = _validar_Xy(X, y, binaria=True)
        yi = y.astype(int)
        N, Nc = len(yi), np.bincount(yi, minlength=2)
        if (Nc == 0).any():
            raise ValueError("El entrenamiento necesita ambas clases para calcular a_c.")
        self.pesos_clase_ = N / (2.0 * Nc)            # se recalculan en cada entrenamiento
        self._a = self.pesos_clase_[yi]
        return self._ajustar(_aumentar(X, self.fit_intercept), y)

    def _objetivo_theta(self, Xa, y, theta):
        z = Xa @ theta
        perdida = np.sum(self._a * (log1pexp(z) - y * z)) / self._a.sum()
        return perdida + 0.5 * self.lambda2 * np.sum(theta[self._i0:] ** 2)

    def _gradiente_suave(self, Xa, y, theta):
        from src.modelos import sigmoide
        r = self._a * (sigmoide(Xa @ theta) - y)
        grad = Xa.T @ r / self._a.sum()
        grad[self._i0:] += self.lambda2 * theta[self._i0:]
        return grad

    def _constante_lipschitz(self, Xa):
        return 0.25 * np.linalg.norm(np.sqrt(self._a)[:, None] * Xa, 2) ** 2 / self._a.sum() + self.lambda2


def crear_modelo(nombre, lam):
    clase = LogisticaGD if nombre == "sin_ponderar" else LogisticaPonderadaGD
    return clase(lambda2=lam, tol=TOL_GD, max_iter=MAX_ITER_GD)


def ajustar_predecir(nombre, lam, X_fit, y_fit, X_eval, prep=None):
    # Si prep es None, la preparación se ajusta con X_fit (correcto). En (c) se pasa ya ajustada.
    prep = construir_prep().fit(X_fit) if prep is None else prep
    modelo = crear_modelo(nombre, lam).fit(prep.transform(X_fit), np.asarray(y_fit))
    if modelo.motivo_ != "convergencia":
        raise RuntimeError(f"{nombre}, λ={lam}: {modelo.motivo_} tras {modelo.n_iter_} iteraciones")
    return modelo.predict_proba(prep.transform(X_eval))[:, 1], modelo

### b.1 Verificación de la equivalencia con scikit-learn

En el entrenamiento completo, para cada $\lambda_2$: coeficientes del modelo propio frente a `LogisticRegression(C=1/(λ₂N))`, sin pesos y con `class_weight="balanced"`.

In [ ]:
prep_full = construir_prep().fit(X_train)
Z_train, yv = prep_full.transform(X_train), y_train.to_numpy()
N = len(yv)
assert np.isclose((N / (2 * np.bincount(yv)))[yv].sum(), N)      # Σ a_{y_n} = N

filas = []
for nombre, cw in [("sin_ponderar", None), ("ponderado", "balanced")]:
    for lam in LAMBDAS:
        propio = crear_modelo(nombre, lam).fit(Z_train, yv)
        ref = LogisticRegression(C=1 / (lam * N), class_weight=cw, tol=1e-12, max_iter=100_000).fit(Z_train, yv)
        filas.append({"modelo": nombre, "λ2": lam, "iteraciones_GD": propio.n_iter_,
                      "máx|Δw|": np.abs(propio.w_ - ref.coef_.ravel()).max(),
                      "|Δb|": abs(propio.b_ - ref.intercept_[0])})
verif = pd.DataFrame(filas)
display(verif)

Las diferencias con scikit-learn son del orden de $10^{-5}$ o menores en coeficientes e intercepto (las mayores, con $\lambda_2 = 10^{-3}$, vienen de la tolerancia del descenso por gradiente, que necesita más iteraciones cuando la penalización es débil). Los dos objetivos tienen el mismo minimizador.

### b.2 Validación cruzada temporal

Cuatro pliegues de entrenamiento expansivo (`TimeSeriesSplit(n_splits=4, gap=24)`). En cada pliegue se ajustan de nuevo la preparación, los pesos $a_c$ y el modelo, solo con el pasado del pliegue. AP se mide sin ponderar ni remuestrear el pliegue evaluado.

In [ ]:
cv_temporal = TimeSeriesSplit(n_splits=4, gap=GAP)
pliegues = list(cv_temporal.split(X_train))

desc = []
for k, (i_fit, i_eval) in enumerate(pliegues, 1):
    assert i_fit.max() + GAP < i_eval.min()                       # hueco de un día
    assert y_train.iloc[i_eval].nunique() == 2, f"pliegue {k} sin ambas clases"
    assert y_train.iloc[i_fit].nunique() == 2
    desc.append({"pliegue": k, "n_fit": len(i_fit), "positivos_fit": int(y_train.iloc[i_fit].sum()),
                 "ids_eval": f"{train['id_vuelo'].iloc[i_eval[0]]}–{train['id_vuelo'].iloc[i_eval[-1]]}",
                 "n_eval": len(i_eval), "positivos_eval": int(y_train.iloc[i_eval].sum())})
display(pd.DataFrame(desc))

In [ ]:
filas_cv = []
for k, (i_fit, i_eval) in enumerate(pliegues, 1):
    Xf, yf = X_train.iloc[i_fit], y_train.iloc[i_fit]
    Xe, ye = X_train.iloc[i_eval], y_train.iloc[i_eval]
    for nombre in ["sin_ponderar", "ponderado"]:
        for lam in LAMBDAS:
            p, _ = ajustar_predecir(nombre, lam, Xf, yf, Xe)
            filas_cv.append({"pliegue": k, "modelo": nombre, "λ2": lam,
                             "AP": average_precision_score(ye, p),
                             "log_loss": log_loss(ye, p, labels=[0, 1])})
    # Referencias: probabilidad constante = prevalencia del entrenamiento del pliegue; no emitir alertas.
    p_const = np.full(len(ye), yf.mean())
    filas_cv.append({"pliegue": k, "modelo": "ref_prevalencia", "λ2": np.nan,
                     "AP": average_precision_score(ye, p_const), "log_loss": log_loss(ye, p_const, labels=[0, 1])})
    filas_cv.append({"pliegue": k, "modelo": "ref_sin_alertas", "λ2": np.nan,
                     "AP": np.nan, "log_loss": np.nan,
                     "recall": recall_score(ye, np.zeros(len(ye)), zero_division=0),
                     "accuracy": 1 - ye.mean()})
cv = pd.DataFrame(filas_cv)

resumen_cv = (cv[cv.modelo.isin(["sin_ponderar", "ponderado"])]
              .groupby(["modelo", "λ2"])
              .agg(AP_media=("AP", "mean"), AP_de=("AP", "std"), log_loss_media=("log_loss", "mean"))
              .reset_index())
display(resumen_cv.pivot(index="λ2", columns="modelo", values=["AP_media", "AP_de", "log_loss_media"]))
display(cv[cv.modelo.str.startswith("ref")].groupby("modelo")[["AP", "log_loss", "recall", "accuracy"]].mean())

- Los cuatro pliegues de evaluación tienen ambas clases (27 a 31 positivos cada uno).
- **Referencias:** la probabilidad constante igual a la prevalencia del entrenamiento del pliegue obtiene AP ≈ 0.026 (la prevalencia evaluada) y log-loss ≈ 0.120. No emitir alertas da accuracy ≈ 0.974 con recall 0: una accuracy alta sin detectar ningún retraso.
- Los dos modelos superan con holgura a la referencia en AP (0.21–0.25 frente a 0.026).
- El modelo ponderado tiene log-loss mucho mayor (≈ 0.46–0.68 frente a ≈ 0.10–0.12): los pesos llevan sus probabilidades hacia una prevalencia efectiva del 50 %. La selección usa AP, que solo depende del orden.

### b.3 Selección

Regla fijada antes de ver resultados: mayor AP media; empates dentro de $10^{-6}$ se resuelven prefiriendo el modelo sin ponderar y, si persiste, el mayor $\lambda_2$.

In [ ]:
def seleccionar(resumen, tol=1e-6):
    mejor = resumen["AP_media"].max()
    empatados = resumen[resumen["AP_media"] >= mejor - tol].copy()
    empatados["prioridad_modelo"] = (empatados["modelo"] != "sin_ponderar").astype(int)
    return empatados.sort_values(["prioridad_modelo", "λ2"], ascending=[True, False]).iloc[0]

ganador = seleccionar(resumen_cv)
MODELO_GANADOR, LAMBDA_GANADOR = ganador["modelo"], ganador["λ2"]
# Candidato final de cada familia (para los diagnósticos de validación en (c))
CANDIDATOS = {m: seleccionar(resumen_cv[resumen_cv.modelo == m])["λ2"] for m in ["sin_ponderar", "ponderado"]}
print(f"Ganador por CV: {MODELO_GANADOR}, λ2 = {LAMBDA_GANADOR}  (AP media = {ganador['AP_media']:.4f})")
print("Mejor λ2 por familia:", CANDIDATOS)

**Resultado.** El ganador es la logística **ponderada con $\lambda_2 = 0.1$** (AP media 0.2465). El mejor modelo sin ponderar es $\lambda_2 = 0.01$ (AP media 0.2357). No hay empate dentro de $10^{-6}$, así que la regla de desempate no interviene.

La diferencia entre ambos (≈ 0.011) es pequeña frente a la dispersión entre pliegues (DE ≈ 0.08–0.10). Esa dispersión describe la variación entre períodos, no un intervalo de confianza; la regla fijada se aplica igual.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for nombre, color in [("sin_ponderar", "#2563eb"), ("ponderado", "#d97706")]:
    d = resumen_cv[resumen_cv.modelo == nombre]
    ax.errorbar(d["λ2"], d["AP_media"], yerr=d["AP_de"], marker="o", capsize=3, label=nombre, color=color)
ax.axhline(cv.loc[cv.modelo == "ref_prevalencia", "AP"].mean(), ls="--", color="gray",
           label="probabilidad constante (prevalencia)")
ax.set(xscale="log", xlabel=r"$\lambda_2$", ylabel="AP media (4 pliegues)",
       title="Selección por AP en CV temporal")
ax.legend()
plt.tight_layout()
plt.show()

---
## (c) Contraexperimento de fuga y diagnósticos en validación

### c.1 Contraexperimento (solo dentro de entrenamiento)

Modelo, $\lambda_2$ y pliegues fijos (los del ganador). Dos procedimientos:
- **Contaminado:** la preparación (medianas, modas, escalas, categorías) se ajusta una vez con todo `X_train`, que incluye el futuro de cada pliegue.
- **Correcto:** la preparación se ajusta de nuevo con el pasado de cada pliegue.

Los resultados contaminados no participan en la selección.

In [ ]:
prep_contaminada = construir_prep().fit(X_train)            # ve todo el entrenamiento
filas_fuga = []
for k, (i_fit, i_eval) in enumerate(pliegues, 1):
    Xf, yf = X_train.iloc[i_fit], y_train.iloc[i_fit]
    Xe, ye = X_train.iloc[i_eval], y_train.iloc[i_eval]
    p_ok, _ = ajustar_predecir(MODELO_GANADOR, LAMBDA_GANADOR, Xf, yf, Xe)
    p_mal, _ = ajustar_predecir(MODELO_GANADOR, LAMBDA_GANADOR, Xf, yf, Xe, prep=prep_contaminada)
    filas_fuga.append({"pliegue": k,
                       "AP_correcto": average_precision_score(ye, p_ok),
                       "AP_contaminado": average_precision_score(ye, p_mal),
                       "logloss_correcto": log_loss(ye, p_ok, labels=[0, 1]),
                       "logloss_contaminado": log_loss(ye, p_mal, labels=[0, 1])})
fuga = pd.DataFrame(filas_fuga)
fuga["ΔAP (contam − correcto)"] = fuga["AP_contaminado"] - fuga["AP_correcto"]
fuga["Δlogloss (contam − correcto)"] = fuga["logloss_contaminado"] - fuga["logloss_correcto"]
display(fuga)
display(fuga.filter(like="Δ").agg(["mean", "min", "max"]))

# Qué información entra en cada caso: estadísticas del primer pliegue
i_fit_1 = pliegues[0][0]
display(pd.DataFrame({"mediana_pasado_pliegue_1": X_train.iloc[i_fit_1][NUMERICAS].median(),
                      "mediana_todo_train": X_train[NUMERICAS].median(),
                      "media_pasado_pliegue_1": X_train.iloc[i_fit_1][NUMERICAS].mean(),
                      "media_todo_train": X_train[NUMERICAS].mean()}))

**Resultado.** Las diferencias son minúsculas y en su mayoría **negativas**: AP contaminada menor en 0.000–0.005 y log-loss contaminada menor en menos de 0.003. Es decir, el procedimiento con fuga no parece "mejor" en AP, aunque usa información del futuro: por ejemplo, la mediana de `trafico_programado` con el pasado del pliegue 1 es 46.2 y con todo el entrenamiento 48.2.

**Por qué el signo o el tamaño de la diferencia no determinan si hay fuga.**
- La fuga es una propiedad del **procedimiento**: hay fuga si la evaluación usa información que no estaría disponible al predecir. Aquí la hay por construcción, porque las medianas y escalas del caso contaminado se calculan con filas posteriores al pliegue evaluado (incluidas las del propio pliegue de evaluación).
- El efecto en la métrica depende de **cuánto cambian esas estadísticas** y de **cuánto le importan al modelo**. Imputar con una mediana algo distinta o reescalar con otra media altera poco una logística regularizada; además, como la deriva de `trafico_programado` es moderada, las estadísticas del pasado y las de todo el entrenamiento son parecidas.
- Con pocos positivos por pliegue, AP y log-loss tienen mucha variabilidad: una diferencia pequeña puede salir positiva o negativa por azar.
- Por eso, una diferencia cercana a cero no prueba ausencia de fuga, y una diferencia favorable al contaminado tampoco la cuantifica: la métrica solo muestra una consecuencia, que en otro conjunto de datos o con otra preparación (por ejemplo, codificación por objetivo o selección de variables) podría ser grande.

### c.2 Diagnósticos en validación

Los dos candidatos finales (el mejor $\lambda_2$ de cada familia) se ajustan con **todo** el entrenamiento y se evalúan en validación. Estos diagnósticos **no cambian el ganador** elegido por CV.

In [ ]:
prob_valid, modelos_finales = {}, {}
for nombre, lam in CANDIDATOS.items():
    prob_valid[nombre], modelos_finales[nombre] = ajustar_predecir(nombre, lam, X_train, y_train, X_valid)

def diagnostico(y, p):
    prec, rec, _ = precision_recall_curve(y, p)
    return {"ROC_AUC": roc_auc_score(y, p), "AP": average_precision_score(y, p),
            "PR_AUC_trapezoidal": auc(rec, prec),
            "log_loss": log_loss(y, p, labels=[0, 1]), "Brier": brier_score_loss(y, p),
            "p_media": p.mean(), "prevalencia_valid": y.mean()}

diag = pd.DataFrame({f"{n} (λ2={CANDIDATOS[n]:g})": diagnostico(y_valid, p) for n, p in prob_valid.items()})
diag["ref_prevalencia_train"] = pd.Series(diagnostico(y_valid, np.full(len(y_valid), y_train.mean())))
display(diag)

**Lectura.**
- **Orden (ROC-AUC, AP):** los dos candidatos son prácticamente iguales en validación (ROC-AUC ≈ 0.85, AP ≈ 0.39). La prevalencia de validación es 3.9 %, así que AP ≈ 0.39 es unas 10 veces la de la referencia constante.
- **Probabilidades (log-loss, Brier):** el ponderado es mucho peor (log-loss 0.535 frente a 0.126; Brier 0.178 frente a 0.032). Su probabilidad media es 0.39 cuando la frecuencia real es 0.039. Ponderar cambia la escala de las probabilidades sin mejorar el orden.
- **AP frente a área trapezoidal:** para los modelos difieren poco (0.394 frente a 0.392). Para la referencia constante la diferencia es enorme (AP = 0.039, trapezoidal = 0.52): la interpolación lineal entre el punto (recall 0, precisión 1) y (recall 1, precisión 0.039) inventa un área que ningún umbral alcanza.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for (nombre, p), color in zip(prob_valid.items(), ["#2563eb", "#d97706"]):
    fpr, tpr, _ = roc_curve(y_valid, p)
    prec, rec, _ = precision_recall_curve(y_valid, p)
    axes[0].plot(fpr, tpr, color=color, label=f"{nombre}: AUC={roc_auc_score(y_valid, p):.3f}")
    axes[1].step(rec, prec, where="post", color=color,
                 label=f"{nombre}: AP={average_precision_score(y_valid, p):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="Tasa de falsos positivos", ylabel="Recall", title="ROC en validación")
axes[1].axhline(y_valid.mean(), ls="--", color="gray", label="prevalencia de validación")
axes[1].set(xlabel="Recall", ylabel="Precisión", title="Precisión–recall en validación")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**AP frente al área trapezoidal bajo PR.** AP $=\sum_k (R_k - R_{k-1})P_k$: suma escalonada que usa la precisión en cada nuevo nivel de recall. El área trapezoidal interpola linealmente entre puntos de la curva PR, lo que es optimista porque la precisión no varía linealmente entre umbrales. Por eso los dos números difieren; se reporta AP como métrica principal.

### c.3 Calibración con cinco grupos por cuantiles

Grupos por cuantiles de la probabilidad predicha en validación. Si hay probabilidades repetidas, algunos cortes coinciden y los grupos se fusionan (`duplicates="drop"`); se reporta el número de grupos efectivos.

In [ ]:
def tabla_calibracion(y, p, grupos=5):
    grupo = pd.qcut(p, q=grupos, duplicates="drop")
    t = (pd.DataFrame({"y": np.asarray(y), "p": p, "grupo": grupo})
           .groupby("grupo", observed=True)
           .agg(n=("y", "size"), positivos=("y", "sum"), p_media=("p", "mean"), frecuencia_obs=("y", "mean")))
    return t, grupo.categories.size

tablas_cal = {}
for nombre, p in prob_valid.items():
    t, efectivos = tabla_calibracion(y_valid, p)
    tablas_cal[nombre] = t
    print(f"{nombre}: {efectivos} grupos efectivos de 5; probabilidades distintas = {np.unique(p).size} de {len(p)}")
    display(t)

fig, ax = plt.subplots(figsize=(6, 5))
for (nombre, t), color in zip(tablas_cal.items(), ["#2563eb", "#d97706"]):
    ax.plot(t["p_media"], t["frecuencia_obs"], "o-", color=color, label=nombre)
    for _, fila in t.iterrows():
        ax.annotate(f"{int(fila.positivos)}/{int(fila.n)}", (fila.p_media, fila.frecuencia_obs),
                    textcoords="offset points", xytext=(4, 4), fontsize=7, color=color)
lim = max(t["p_media"].max() for t in tablas_cal.values()) * 1.05
ax.plot([0, lim], [0, lim], "--", color="gray", label="calibración perfecta")
ax.set(xscale="symlog", yscale="symlog", xlabel="probabilidad media predicha", ylabel="frecuencia observada",
       title="Calibración en validación (5 grupos por cuantiles)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Ganador elegido por CV (sin cambios): {MODELO_GANADOR}, λ2 = {LAMBDA_GANADOR}")

**Calibración.**
- Las 1896 probabilidades de cada modelo son distintas, así que los 5 grupos son efectivos y tienen 379–380 casos. Con solo 74 positivos en validación, los grupos bajos tienen 0–10 positivos y su frecuencia observada es muy imprecisa.
- **Sin ponderar:** sigue la diagonal en los grupos bajos y subestima en el último (media predicha 0.090, frecuencia observada 0.140).
- **Ponderado:** sobrestima en todos los grupos (por ejemplo, 0.70 predicho frente a 0.14 observado en el último). Su probabilidad no puede leerse como probabilidad de retraso sin recalibrar.
- **El ganador no cambia:** sigue siendo el ponderado con $\lambda_2 = 0.1$, elegido por AP en CV. Su mala calibración importa al elegir el umbral de la política: el umbral tendrá que fijarse sobre la escala de sus probabilidades, no sobre la prevalencia real.